# Día 6 — Visualización, optimización y proyecto

Conectar PCA con evaluación honesta y búsqueda de hiperparámetros.

## Objetivos de aprendizaje

- Comunicar un conjunto multidimensional mediante PCA.
- Construir baseline y optimizar con validación cruzada agrupada.
- Evitar mezclar países entre entrenamiento y prueba.
- Elegir un dataset pertinente y preparar un proyecto reproducible.

> **Tiempo del plan oficial:** 3 horas sincrónicas + 1 hora autónoma. Prioriza visualización, una búsqueda pequeña y el diseño/comparación inicial del proyecto; deja búsquedas extensas como práctica autónoma.

**Dataset:** indicadores país-año del Banco Mundial descargados localmente en `datos/world_bank_latam_ml.csv`. La celda de búsqueda no accede a servicios externos.

## Cómo estudiar esta jornada

Lee primero la pregunta y la explicación; ejecuta luego las celdas en orden. Después de cada tabla o gráfico, detente en **Interpreta** y redacta una conclusión con evidencia. Los resultados dependen de la versión de las librerías y de las particiones; usa tus propios números y no copies los ejemplos como verdades universales. No confundas predicción con causalidad.

## 1. Visualización multidimensional como puente a PCA

Retomamos los indicadores de los días 4–5. Un gráfico bidimensional no puede mostrar seis dimensiones originales. PCA proyecta la variación sobre dos ejes; usaremos el color para inspeccionar crecimiento del PIB, no como prueba supervisada ni como predictor causal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

macro = pd.read_csv(DATA_DIR / "world_bank_latam_ml.csv")
features = ["inflacion_pct", "desempleo_pct", "exportaciones_pct_pib",
            "inversion_pct_pib", "crecimiento_poblacion_pct"]
target = "crecimiento_pib_pct"
df = macro.dropna(subset=features+[target]).copy()
Xz = StandardScaler().fit_transform(df[features])
coords = PCA(n_components=2, random_state=42).fit_transform(Xz)
plt.figure(figsize=(8,5))
sc = plt.scatter(coords[:,0], coords[:,1], c=df[target], cmap="coolwarm", alpha=.8)
plt.colorbar(sc,label="Crecimiento PIB (%)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Vista PCA de indicadores macroeconómicos")


### Interpreta visualización

Describe agrupaciones, años extremos y solapamiento; revisa loadings de Día 5 para dar significado a los ejes. PCA se ajustó aquí a toda la tabla para una visualización exploratoria; esa figura **no** se reutiliza para estimar desempeño predictivo. Para modelar, cada transformación se ajustará dentro de los folds.

## 2. El problema de optimización

Un hiperparámetro (`max_depth`, `min_samples_leaf`) se fija antes de ajustar el modelo; los parámetros se aprenden de los datos. Probar muchas configuraciones sobre el mismo Validation puede sobreajustar también la selección. Usaremos una validación cruzada interna y compararemos con una referencia sencilla.

Para evitar mezclar un mismo país entre Train y Test, asignaremos países completos con una semilla reproducible y verificaremos la separación. Con solo siete países, la estimación tendrá mucha incertidumbre y no debe venderse como evaluación definitiva.

In [ ]:
paises = np.array(sorted(df.iso3.unique()))
paises = np.random.default_rng(42).permutation(paises)
n_test = max(2, round(.25 * len(paises)))
paises_test = set(paises[:n_test])
mask_test = df.iso3.isin(paises_test)
X_train, y_train = X.loc[~mask_test], y.loc[~mask_test]
X_test, y_test = X.loc[mask_test], y.loc[mask_test]
g_train = df.loc[~mask_test, "iso3"]
print("Países Train:", sorted(set(g_train)))
print("Países Test:", sorted(paises_test))
assert set(g_train).isdisjoint(set(df.loc[mask_test, "iso3"]))
print("Tamaños:", X_train.shape, X_test.shape)


## 3. Baseline y búsqueda de hiperparámetros

El baseline predice la media de crecimiento del Train. El modelo será Random Forest Regressor; comparamos profundidad y tamaño mínimo de hoja. GroupKFold mantiene países completos separados dentro de cada fold. El escalado no es necesario para árboles, pero el Pipeline mantiene el diseño listo para agregar transformaciones sin leakage.

In [ ]:
cv = GroupKFold(n_splits=min(3, g_train.nunique()))
baseline = DummyRegressor(strategy="mean")
scores_baseline = cross_val_score(baseline, X_train, y_train, groups=g_train, cv=cv,
                                  scoring="neg_mean_absolute_error", n_jobs=-1)
mae_baseline_cv = -scores_baseline.mean()
print("MAE CV baseline, puntos porcentuales:", round(mae_baseline_cv, 3))

pipe = Pipeline([("model", RandomForestRegressor(n_estimators=150, random_state=42))])
busqueda = GridSearchCV(pipe,
    param_grid={"model__max_depth":[2,4,None], "model__min_samples_leaf":[2,5,10]},
    scoring="neg_mean_absolute_error", cv=cv, refit=True, n_jobs=-1,
    return_train_score=True)
busqueda.fit(X_train, y_train, groups=g_train)
print("Mejores parámetros:", busqueda.best_params_)
print("MAE CV medio:", round(-busqueda.best_score_, 3))


### Interpreta la búsqueda

`best_score_` es negativo porque scikit-learn maximiza scores; cambiamos el signo para leer MAE. Compara MAE de CV con el baseline y `cv_results_` (media y desviación por configuración). Una diferencia pequeña frente a la variabilidad no justifica complejidad adicional. GroupKFold con pocos países genera folds distintos y resultados frágiles; reporta cantidad de grupos y no generalices a todo Ecuador.

In [ ]:
cv_tabla = pd.DataFrame(busqueda.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf",
    "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]].copy()
cv_tabla["MAE_train"] = -cv_tabla.pop("mean_train_score")
cv_tabla["MAE_CV"] = -cv_tabla.pop("mean_test_score")
cv_tabla = cv_tabla.sort_values("rank_test_score")
cv_tabla.round(3)


## 4. Evaluación de Test una sola vez

La configuración se seleccionó exclusivamente mediante CV sobre Train. Ahora medimos Test por países no vistos. No regresamos a cambiar hiperparámetros tras ver el resultado. Si lo hacemos, necesitamos un nuevo conjunto independiente.

In [ ]:
if -busqueda.best_score_ < mae_baseline_cv:
    modelo = busqueda.best_estimator_
    nombre_modelo = "Random Forest"
else:
    modelo = DummyRegressor(strategy="mean").fit(X_train, y_train)
    nombre_modelo = "Baseline media"
pred = modelo.predict(X_test)
pd.DataFrame([{"modelo":nombre_modelo, "MAE":mean_absolute_error(y_test,pred), "R2":r2_score(y_test,pred)}]).set_index("modelo").round(3)


### Interpretación ejecutiva

MAE está en puntos porcentuales. Compara RF contra baseline: si no lo supera de forma clara, la conclusión correcta es que no hay evidencia de mejora con esta muestra y diseño. $R^2$ negativo en Test puede ocurrir, especialmente al dejar países completos fuera. Un buen resultado tampoco demuestra causalidad ni garantiza transferibilidad al BCE.

## 5. Búsqueda aleatoria, visualización y elección de datos

`GridSearchCV` explora todas las combinaciones indicadas; una cuadrícula amplia cuesta mucho. `RandomizedSearchCV` muestrea una cantidad fijada y es útil para espacios grandes. En esta práctica de 3 horas, la cuadrícula pequeña permite inspeccionar cada resultado y evita distraerse con una búsqueda costosa. Para el proyecto final prioriza: fuente descargada y documentada, unidad/target claros, tamaño suficiente, ausencia de datos confidenciales y baseline.

## Taller U4 — Reto de optimización (20 puntos sugeridos)

Entrega: (1) figura PCA con lectura de ejes; (2) baseline; (3) búsqueda con CV apropiada y tabla de al menos seis configuraciones; (4) selección justificada con promedio y variabilidad; (5) Test solo al final; (6) límites de la muestra y recomendación prudente. Puedes proponer un dataset económico alternativo solo si se descarga y se guarda en `datos/` con fuente, fecha, licencia y diccionario.

### Preguntas de cierre

¿Por qué se agrupan países al separar datos? ¿Qué evita CV frente a mirar una sola partición? ¿Por qué una búsqueda de muchos hiperparámetros puede sobreajustar? ¿Qué significa superar al baseline? ¿Por qué la figura de PCA de toda la muestra no es parte del pipeline predictivo?